In [1]:
import os
import tifffile
import rasterio

import cv2
import numpy as np

import leafmap.leafmap as leafmap
#from samgeo import SamGeo2

import geopandas as gpd
import pickle
from pyproj import Transformer

from utils.raster_tools import Raster_profile #class type for raster profile
import matplotlib.pyplot as plt

from utils.tools import setup_polygon, save_stats, read_npz

### See the overall

In [ ]:
clipped_theos_file = "theos/clipped_IMG_T2V_20250119034323_ORTHO_PMS_32_small.tif"

# m = leafmap.Map(center=[(lat_start + lat_end)/2, (long_start + long_end)/2], zoom=16, height="800px")
m = leafmap.Map(center=[12.908807, 100.922147], zoom=16 , height="800px") 

m.add_basemap("Satellite") 
m.add_raster(clipped_theos_file, layer_name="Theos") 
m

## Read the cliped Google image

In [ ]:
theos_Profile = Raster_profile(clipped_theos_file) 

### Specify the grid's spacing

In [ ]:
long_start, lat_start = theos_Profile.get_longlat_from_image_pixels(0, 0, crs_dst="EPSG:4326")
long_end, lat_end     = theos_Profile.get_longlat_from_image_pixels(7001, 7053, crs_dst="EPSG:4326")

In [ ]:
zoom_level = 18 # Google resolution // the higher zoom level >> higher resolution 
lat_diff   = np.abs(lat_end - lat_start)/10
long_diff  = lat_diff
crs_source = "EPSG:4326" # Google 
crs_target = "EPSG:32647" # Theos

In [ ]:
slice_row   = 5
slice_column = 3

slices_path = "ISP0704-Zoom%d" % zoom_level


slice_subpath  = os.path.join(slices_path, "%000d-%000d" % (slice_row, slice_column))
slice_google_filename = os.path.join(slice_subpath, "google.tif") 
warped_slice_google_filename = os.path.join(slice_subpath, "warped_google.tif") 
slice_theos_filename  = os.path.join(slice_subpath, "theos.tif") 
npz_filename  = os.path.join(slice_subpath, "stats.npz") 
csv_filename = os.path.join(slice_subpath, "stats.csv") 

os.makedirs(slices_path, exist_ok=True)
os.makedirs(slice_subpath, exist_ok=True)


long_start_temp = long_start  + (slice_column-1)*long_diff.item()  
# lat_start_temp  = lat_start  - (slice_no)*lat_diff.item()
lat_start_temp  = lat_start - (slice_row-1)*lat_diff.item()

long_end_temp = long_start + (slice_column)*long_diff.item()  
#lat_end_temp  = lat_start  - (slice_no+1)*lat_diff.item()
lat_end_temp  = lat_start  -  (slice_row)*lat_diff.item()


print("Slice no. %000d-%000d" % (slice_row, slice_column)) 
print("Start: LON: %f LAT: %f" % (long_start_temp, lat_start_temp)) 
print("End  : LON: %f LAT: %f" % (long_end_temp, lat_end_temp)) 

poly_gons, bbox, coordinates = setup_polygon(long_start_temp, lat_start_temp, long_end_temp, lat_end_temp, crs_source=crs_source, crs_target=crs_target)

leafmap.map_tiles_to_geotiff(output=slice_google_filename, bbox=bbox, zoom=zoom_level, source="Satellite", overwrite=True)
leafmap.clip_image(clipped_theos_file, poly_gons, slice_theos_filename)


stats = {"slice_row": slice_row, 
        "slice_column": slice_column,
        "slice_theos_filename": slice_theos_filename,
        "slice_google_filename": slice_google_filename,  
        "long_start_temp": long_start_temp,
        "lat_start_temp": lat_start_temp,
        "long_end_temp": long_end_temp,
        "lat_end_temp": lat_end_temp,
        "poly_gons": poly_gons,
        "bbox":bbox,
        "coordinates": coordinates,
        "crs_source": crs_source,
        "crs_target": crs_target,
        "long_diff": long_diff,
        "lat_diff": lat_diff}

save_stats(stats, npz_filename, csv_filename)

## Test sliced data

### Stats reading

In [ ]:
read_dict = read_npz(npz_filename)
read_dict

### Map reading

In [ ]:
m = leafmap.Map()    
slice_google_filename_prev = 'ISP0704-Zoom18/1-1/google.tif'
slice_theos_filename_prev = 'ISP0704-Zoom18/1-1/theos.tif'
m.add_raster(slice_google_filename_prev, layer_name="Google-prev")
m.add_raster(slice_theos_filename_prev, layer_name="Theos-prev") 
m.add_raster(slice_google_filename, layer_name="Google")
m.add_raster(slice_theos_filename, layer_name="Theos") 
m

### Test warping

In [ ]:
from plantcv import plantcv as pcv
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import tifffile
import os

from utils.tools import get_raster_data, image_enhancement, save_raster_and_write_meta
from utils.tools import setup_polygon,save_stats,read_npz

clipped_theos_file = "theos/clipped_IMG_T2V_20250119034323_ORTHO_PMS_32_small.tif"

slice_row = 5
slice_column = 3
zoom_level = 18
slices_path = "ISP0704-Zoom%d" % zoom_level

In [ ]:
# slice_no   = 0

# slices_path     = "ISP0704-Zoom%d" % zoom_level
slice_subpath  = os.path.join(slices_path, "%000d-%000d" % (slice_row, slice_column))

slice_google_filename = os.path.join(slice_subpath, "google.tif") 
warped_slice_google_filename = os.path.join(slice_subpath, "warped_google.tif") 
slice_theos_filename  = os.path.join(slice_subpath, "theos.tif")  
path_warp_npz  = os.path.join(slice_subpath, "warp_stats.npz")  
path_warp_csv  = os.path.join(slice_subpath, "warp_stats.csv")   


In [ ]:
imgA, _ = get_raster_data(slice_google_filename) 
imgA = np.transpose(imgA, (1, 2, 0))  # Convert from (bands, height, width) to (height, width, bands)
 
imgB, _  = get_raster_data(slice_theos_filename) 
imgB = np.transpose(imgB, (1, 2, 0))  # Convert from (bands, height, width) to (height, width, bands)
imgB = imgB[:,:,:3]
imgB = image_enhancement(imgB) #แก้ exposure intensity (ไว้เพิ่มใน report)

In [ ]:
from utils.interactive_tools import Find_correspondences

%matplotlib widget
marker_AB = Find_correspondences(imgA, imgB, figsize=(13, 5))

In [ ]:
import cv2

point_src  = np.array(marker_AB.points[0])
point_dst  = np.array(marker_AB.points[1])

Homography, status = cv2.findHomography(point_src, point_dst) 

In [ ]:
target_size = (imgB.shape[1], imgB.shape[0])
im_dst = cv2.warpPerspective(imgA, Homography, target_size) 

In [ ]:
warp_stats = {
    "point_src": point_src,
    "point_dst": point_dst,
    "Homography": Homography,
    "target_size": target_size, 
    "src_img_filename": slice_google_filename,
    "dst_img_filename": slice_theos_filename,
    "result_image_filename": warped_slice_google_filename
}

In [ ]:
save_stats(warp_stats, path_warp_npz, path_warp_csv)

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(10,5)) 
axs[0].imshow(imgA[:,:,0:3])  # Display the first three channels (RGB) of the clipped image
axs[0].set_title('Google') # Set a title for the first subplot 

axs[1].imshow(imgB[:,:,0:3])  # Display the first three channels (RGB) of the clipped satellite image
axs[1].set_title('Theos') # Set a title for the second subplot 

axs[2].imshow(im_dst[:,:,0:3])  # Display the first three channels (RGB) of the clipped satellite image
axs[2].set_title('Google (warped)') # Set a title for the second subplot 
 
fig.tight_layout()

In [ ]:
im_dst_4D = np.zeros((imgB.shape[0], imgB.shape[1], 4)) 
im_dst_4D[:,:,:3] = im_dst 
im_dst_4D[:,:, 3] = 254
im_dst_4D = im_dst_4D.transpose(2, 0, 1)
im_dst_4D = im_dst_4D.astype(np.uint8)

destination_tif = warped_slice_google_filename
meta_source_tif = slice_theos_filename
save_raster_and_write_meta(im_dst_4D , destination_tif, meta_source_tif)

In [ ]:
import leafmap.leafmap as leafmap
m = leafmap.Map()
m.add_raster(slice_google_filename, layer_name="Google") 
m.add_raster(slice_theos_filename, layer_name="theos") 
m.add_raster(warped_slice_google_filename, layer_name="Google (warped)")  
m

### Test SamGeo2

In [ ]:
# Resatrt 1 T

In [3]:
import os 
import cv2
import rasterio

import numpy as np

#from samgeo import SamGeo2

import geopandas as gpd
import pickle
from pyproj import Transformer

import matplotlib.pyplot as plt

import leafmap.leafmap as leafmap 
 
from plantcv import plantcv as pcv
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import tifffile 


from utils.raster_tools import Raster_profile #class type for raster profile
from utils.tools import get_raster_data, image_enhancement, save_raster_and_write_meta
from utils.tools import setup_polygon,save_stats,read_npz


clipped_theos_file = "theos/clipped_IMG_T2V_20250119034323_ORTHO_PMS_32_small.tif"

slice_row       = 4
slice_column    = 3
zoom_level      = 18
slices_path     = "ISP0704-Zoom%d" % zoom_level
slice_subpath   = os.path.join(slices_path, "%000d-%000d" % (slice_row, slice_column)) 
warped_slice_google_filename = os.path.join(slice_subpath, "warped_google.tif")  
target_dir      = os.path.join(slice_subpath, "samgeo2mask")
os.makedirs(target_dir, exist_ok=True)

slice_theos_filename  = os.path.join(slice_subpath, "theos.tif") 


In [ ]:
# slice_no   = 0

# slices_path     = "ISP0704-Zoom%d" % zoom_level
# slice_subpath   = os.path.join(slices_path, "%000d-%000d" % (slice_row, slice_column)) 
# warped_slice_google_filename = os.path.join(slice_subpath, "warped_google.tif")  
# target_dir      = os.path.join(slice_subpath, "samgeo2mask")  
# os.makedirs(target_dir, exist_ok=True)


In [4]:
stats_          = read_npz(os.path.join(slice_subpath, "stats.npz") )
lat_start_temp  = stats_["lat_start_temp"]
long_start_temp = stats_["long_start_temp"]

lat_end_temp  = stats_["lat_end_temp"]
long_end_temp = stats_["long_end_temp"]

crs_source = "EPSG:4326" # Google 
crs_target = "EPSG:32647" # Theos

lat_end_half = lat_start_temp - np.abs(lat_end_temp - lat_start_temp)*0.5

warped_slice_google_filename_half_top = os.path.join(slice_subpath, "warped_google_half_top.tif") 
poly_gons_half_top, bbox_half_top, coordinates_half_top = setup_polygon(long_start_temp, lat_start_temp, long_end_temp, lat_end_half, crs_source=crs_source, crs_target=crs_target)
leafmap.clip_image(warped_slice_google_filename, poly_gons_half_top, warped_slice_google_filename_half_top)

target_dir_top      = os.path.join(slice_subpath, "samgeo2mask_top")
os.makedirs(target_dir_top, exist_ok=True)

warped_slice_google_filename_half_bottom = os.path.join(slice_subpath, "warped_google_half_bottom.tif") 
poly_gons_half_bottom, bbox_half_bottom, coordinates_half_bottom = setup_polygon(long_start_temp, lat_end_half, long_end_temp, lat_end_temp, crs_source=crs_source, crs_target=crs_target)
leafmap.clip_image(warped_slice_google_filename, poly_gons_half_bottom, warped_slice_google_filename_half_bottom)

target_dir_bottom      = os.path.join(slice_subpath, "samgeo2mask_bottom")
os.makedirs(target_dir_bottom, exist_ok=True)

Reading input: c:\Users\user\Documents\Gistda_workspace\geo\Building_detection\ISP0704-Zoom18\4-3\warped_google_half_top.tif

Updating dataset tags...
Writing output to: c:\Users\user\Documents\Gistda_workspace\geo\Building_detection\ISP0704-Zoom18\4-3\warped_google_half_top.tif
Reading input: c:\Users\user\Documents\Gistda_workspace\geo\Building_detection\ISP0704-Zoom18\4-3\warped_google_half_bottom.tif

Updating dataset tags...
Writing output to: c:\Users\user\Documents\Gistda_workspace\geo\Building_detection\ISP0704-Zoom18\4-3\warped_google_half_bottom.tif


In [5]:
warped_slice_google_filename_half_top = os.path.join(slice_subpath, "warped_google_half_top.tif") 
# poly_gons_half_top, bbox_half_top, coordinates_half_top = setup_polygon(long_start_temp, lat_start_temp, long_end_temp, lat_end_half, crs_source=crs_source, crs_target=crs_target)
# leafmap.clip_image(warped_slice_google_filename, poly_gons_half_top, warped_slice_google_filename_half_top)

# target_dir_top      = os.path.join(slice_subpath, "samgeo2mask_top")
# os.makedirs(target_dir_top, exist_ok=True)

In [6]:
warped_slice_google_filename_half_bottom = os.path.join(slice_subpath, "warped_google_half_bottom.tif") 
# poly_gons_half_bottom, bbox_half_bottom, coordinates_half_bottom = setup_polygon(long_start_temp, lat_end_half, long_end_temp, lat_end_temp, crs_source=crs_source, crs_target=crs_target)
# leafmap.clip_image(warped_slice_google_filename, poly_gons_half_bottom, warped_slice_google_filename_half_bottom)

# target_dir_bottom      = os.path.join(slice_subpath, "samgeo2mask_bottom")
# os.makedirs(target_dir_bottom, exist_ok=True)

In [ ]:
m = leafmap.Map(center=[12.908807, 100.922147], zoom=18 , height="800px") 
m.add_raster(warped_slice_google_filename, layer_name="warped_google.tif") 
m.add_raster(warped_slice_google_filename_half_top, layer_name="half_top") 
m.add_raster(warped_slice_google_filename_half_bottom, layer_name="half_bottom") 
m

In [ ]:
# restart 1 T

In [ ]:
from samgeo import SamGeo2

sam = SamGeo2(
    model_id="sam2-hiera-large",
    automatic=False,
    device="cuda"
) 


In [ ]:
print("PLEASE SAVE THE MASK under Folder: %s" % target_dir_top)

In [ ]:
sam.set_image(warped_slice_google_filename_half_top)
sam.show_map()

In [ ]:
sam.set_image(warped_slice_google_filename_half_bottom)
sam.show_map()

In [ ]:
mask_filename_top = os.path.join(slice_subpath,"samgeo2mask", "masks-top.tif") 

In [ ]:
mask_filename_bottom = os.path.join(slice_subpath,"samgeo2mask", "masks-bottom.tif") 

In [ ]:
import leafmap.leafmap as leafmap
m = leafmap.Map() 
m.add_raster(slice_theos_filename, layer_name="theos") 
m.add_raster(warped_slice_google_filename, layer_name="Google (warped)")  
m.add_raster(mask_filename_top, cmap="jet", layer_name="Mask (top)")  
m.add_raster(mask_filename_bottom, cmap="jet", layer_name="Mask (bottom)")  
m

In [9]:
import rasterio
from utils.mask_tools import Mask_profile
from rasterio.merge import merge
import pandas as pd
import geopandas as gpd
import os

mask_filename_top  = os.path.join(slice_subpath, "samgeo2mask", "masks-top.tif") 
center_geojson_top = os.path.join(slice_subpath, "samgeo2mask", "masks-top_fg_markers.geojson")   
bb_filename_top    = os.path.join(slice_subpath, "samgeo2mask", "boundbox-top.geojson") 

mask_filename_bottom  = os.path.join(slice_subpath, "samgeo2mask", "masks-bottom.tif") 
center_geojson_bottom = os.path.join(slice_subpath, "samgeo2mask", "masks-bottom_fg_markers.geojson") 
bb_filename_bottom    = os.path.join(slice_subpath, "samgeo2mask", "boundbox-bottom.geojson") 

edited_mask_filename = os.path.join(slice_subpath, "samgeo2mask", "masks-bottom_edited.tif") 


Mask_obj_top = Mask_profile(mask_filename_top, center_geojson_file=center_geojson_top, center_geojson_crs="EPSG:4326")
object_id = np.unique(Mask_obj_top.mask)
num_obs   = max(object_id).item() 

Mask_obj_bottom = Mask_profile(mask_filename_bottom, center_geojson_file=center_geojson_bottom, center_geojson_crs="EPSG:4326")
mask2D = Mask_obj_bottom.mask
mask2D[mask2D > 0] = mask2D[mask2D > 0] + num_obs

# update mask   
meta_mask_filename   = mask_filename_bottom
mask_2D              = mask2D.reshape(1, mask2D.shape[0], mask2D.shape[1])
save_raster_and_write_meta(mask_2D, edited_mask_filename, meta_mask_filename)

# update boundingbox
# Mask_obj_bottom_edited = Mask_profile(edited_mask_filename, center_geojson_file=center_geojson_bottom, center_geojson_crs="EPSG:4326")
# gdf_bottom = Mask_obj_bottom_edited.make_boundboxes()
# bb_filename = os.path.join(slice_subpath, "samgeo2mask", "boundbox-bottom.geojson")  
# gdf_bottom.to_file(bb_filename, driver='GeoJSON')  

# merge geo tif 
tiff_files = [mask_filename_top, edited_mask_filename]

src_files = [rasterio.open(fp) for fp in tiff_files]
mosaic, out_trans = merge(src_files)

out_meta = src_files[0].meta.copy()
out_meta.update({
    "driver": "GTiff", "height": mosaic.shape[1],
    "width": mosaic.shape[2], "transform": out_trans,
    "compress": "lzw" 
})

# Write output
output_file  = os.path.join(slice_subpath, "samgeo2mask", "masks-merge.tif") 

with rasterio.open(output_file, 'w', **out_meta) as dest:
    dest.write(mosaic)

for src in src_files: src.close()


# merge center geopandas  
gdf_top    = gpd.read_file(center_geojson_top) 
gdf_bottom = gpd.read_file(center_geojson_bottom)
rdf =pd.concat([gdf_top, gdf_bottom], ignore_index=True)
merged_center_filename = os.path.join(slice_subpath, "samgeo2mask", "center-merge.geojson") 
rdf.to_file(merged_center_filename, driver='GeoJSON')  

Dataset name: ISP0704-Zoom18\4-3\samgeo2mask\masks-top.tif
File mode: r
Number of bands: 1
Image width: 696 pixels
Image height: 358 pixels
Coordinate Reference System (CRS): EPSG:32647
Data shape: (1, 358, 696)
Data type: int32
Dataset name: ISP0704-Zoom18\4-3\samgeo2mask\masks-bottom.tif
File mode: r
Number of bands: 1
Image width: 695 pixels
Image height: 359 pixels
Coordinate Reference System (CRS): EPSG:32647
Data shape: (1, 359, 695)
Data type: int32
Modified image saved to : ISP0704-Zoom18\4-3\samgeo2mask\masks-bottom_edited.tif


In [10]:
Mask_obj_bottom_edited = Mask_profile(output_file, center_geojson_file=merged_center_filename, center_geojson_crs="EPSG:4326")
gdf_bottom = Mask_obj_bottom_edited.make_boundboxes()
bb_filename = os.path.join(slice_subpath, "samgeo2mask", "boundbox-merge.geojson")  
gdf_bottom.to_file(bb_filename, driver='GeoJSON')  

Dataset name: ISP0704-Zoom18\4-3\samgeo2mask\masks-merge.tif
File mode: r
Number of bands: 1
Image width: 698 pixels
Image height: 711 pixels
Coordinate Reference System (CRS): EPSG:32647
Data shape: (1, 711, 698)
Data type: int32


In [ ]:
m = leafmap.Map()
m.add_raster(warped_slice_google_filename, layer_name="Image") 
m.add_circle_markers_from_xy(merged_center_filename, radius=3, color="red", fill_color="yellow", fill_opacity=0.8
)   
m.add_raster(output_file, cmap="jet", layer_name="Building masks")   
m.add_vector(bb_filename, layer_name="Bounding Boxes")
m

Map(center=[12.931360999999999, 100.894423], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_…

In [22]:
m = leafmap.Map(center=[12.908807, 100.922147], zoom=18 , height="800px") 
m.add_raster(warped_slice_google_filename, layer_name="warped_google.tif") 
m.add_raster(os.path.join(slice_subpath, "samgeo2mask", "masks-merge.tif") , layer_name="merge") 
m

Map(center=[12.931360999999999, 100.894423], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_…

## Mask improvement

In [ ]:
from utils.mask_tools import Mask_profile

from plantcv import plantcv as pcv
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import tifffile

from utils.tools import get_raster_data, image_enhancement, save_raster_and_write_meta

In [ ]:
#slice_no =0

mask_filename = os.path.join(slice_subpath, "samgeo2mask", "masks.tif") 
center_geojson = os.path.join(slice_subpath, "samgeo2mask", "masks_fg_markers.geojson")  
warped_slice_google_filename = os.path.join(slice_subpath, "warped_google.tif") 
sat_image, _ = get_raster_data(warped_slice_google_filename)

In [ ]:
#slice_no =0

mask_filename = os.path.join(slice_subpath, "samgeo2mask", "masks-bottom.tif") 
center_geojson = os.path.join(slice_subpath, "samgeo2mask", "masks_fg_markers.geojson")  
warped_slice_google_filename = os.path.join(slice_subpath, "warped_google.tif") 
sat_image, _ = get_raster_data(warped_slice_google_filename)

In [ ]:
bb_filename = os.path.join(slice_subpath, "samgeo2mask", "boundbox.geojson") 
 
gdf = Mask_obj.make_boundboxes()
gdf.to_file(bb_filename, driver='GeoJSON') 

In [ ]:
m = leafmap.Map()
m.add_raster(warped_slice_google_filename, layer_name="Image") 
m.add_circle_markers_from_xy(center_geojson, radius=3, color="red", fill_color="yellow", fill_opacity=0.8
)  
# ถ้า edit มีปัญหา
m.add_raster(mask_filename, cmap="jet", layer_name="Building masks (before)")  
# ถ้า edit ไม่มีปัญหา
#m.add_raster(edited_mask_filename, cmap="jet", layer_name="Building masks (after)")  
m.add_vector(bb_filename, layer_name="Bounding Boxes")
m